# Imports

In [1]:
%matplotlib inline

In [3]:
#importing necessary libraries
import matplotlib.pyplot as plt
import numpy as np
from mplsoccer import Pitch, Sbopen, VerticalPitch
import pandas as pd
from scipy.stats.contingency import odds_ratio
from warnings import filterwarnings
filterwarnings('ignore')

# Select matches from each team in EURO2024

In [46]:
list_euro2004_matches = [3942819, 3943043, 3942752, 3942382, 3942349, 3930180, 3930171, 3942227, 3942226, 3938645, 3930184, 3941022, 3941021, 3941020, 3941019, 3941018, 3941017, 3930182, 3930179, 3940983, 3940878, 3930177, 3930173, 3930172, 3930167, 3930168, 3930165, 3930164, 3930161, 3938637, 3938640, 3938642, 3938639, 3938641, 3938644, 3938643, 3938638, 3930183, 3930181, 3930178, 3930176, 3930175, 3930174, 3930170, 3930169, 3930166, 3930163, 3930162, 3930160, 3930159, 3930158]

In [54]:
parser=Sbopen()
teams_names=[]
for match in list_euro2004_matches:
    list_teams_match=parser.event(match)[0].team_name.unique()
    for team in list_teams_match:
        if team not in teams_names:
            teams_names.append(team)

In [62]:
def list_team_matches(team_name="Portugal"):
    parser = Sbopen()
    list_matches=[match for match in list_euro2004_matches if team_name in  parser.event(match)[0].team_name.unique()]
    return list_matches

In [68]:
list_teams_matches=[(team,list_team_matches(team_name=team)) for team in teams_names]

In [119]:
df_teams_matches=pd.DataFrame(list_teams_matches,columns=("Team","Matches"))

# Team UPPOR per match

In [115]:
def uppor_team(team_name=team):
    odds = []
    for match in df_teams_matches[df_teams_matches.Team==team_name].Matches.to_list()[0]:
        parser = Sbopen()
        df= parser.event(match)[0]
        passes = df.loc[df['team_name'] == team_name].loc[df['type_name'] == 'Pass'].loc[df['sub_type_name'] != 'Throw-in'].loc[df['sub_type_name'] != 'Corner'].loc[df['sub_type_name'] != 'Free Kick'].loc[df['sub_type_name'] != 'Kick Off'].loc[df['sub_type_name'] != 'Goal Kick'].set_index('id')  
        df_pass = passes[['x', 'y', 'end_x', 'end_y','under_pressure','pass_length']]
        df_pass_long=df_pass.loc[df_pass.pass_length>10.9361]
        df_pass_not_long=df_pass.loc[df_pass.pass_length<=10.9361]
        df_pass_pressure=df_pass.loc[df_pass.under_pressure==1]
        df_pass_not_pressure=df_pass.loc[df_pass.under_pressure!=1]
        df_pass_long_pressure=df_pass_long.loc[df_pass_long.under_pressure==1]
        df_pass_long_not_pressure=df_pass_long.loc[df_pass_long.under_pressure!=1]
        df_pass_not_long_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure==1]
        df_pass_not_long_not_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure!=1]
        res = odds_ratio([[len(df_pass_long_pressure),len(df_pass_long_not_pressure)],[len(df_pass_not_long_pressure),len(df_pass_not_long_not_pressure)]],kind='sample')
        number = [[len(df_pass_long_pressure),len(df_pass_long_not_pressure)],[len(df_pass_not_long_pressure),len(df_pass_not_long_not_pressure)]]
        odds.append((match,number,res.statistic,res.confidence_interval(confidence_level=0.95)))
    df_odds=pd.DataFrame(odds,columns=["Jogo","Passes","UPPOR","Intervalo de confiança"])
    return df_odds

In [117]:
uppor_team(team_name="Portugal")

,Jogo,Passes,UPPOR,Intervalo de confiança
0,3942349,"[[48, 607], [14, 200]]",1.129678,"(0.6098918602851245, 2.09245523339237)"
1,3941020,"[[76, 506], [33, 166]]",0.755540,"(0.48435237543300774, 1.178563571339416)"
2,3938644,"[[36, 448], [19, 200]]",0.845865,"(0.47348754889176514, 1.5110999803688998)"
3,3930174,"[[64, 357], [28, 119]]",0.761905,"(0.4666871094378811, 1.243871652920424)"
4,3930166,"[[60, 487], [27, 133]]",0.606890,"(0.3706747291274969, 0.9936360806058636)"


# Team UPPOR all matches

In [122]:
def uppor_team_all(team_name=team):
    odds = []
    passes_long_pressure=0
    passes_long_not_pressure=0
    passes_not_long_pressure=0
    passes_not_long_not_pressure=0
    for match in df_teams_matches[df_teams_matches.Team==team_name].Matches.to_list()[0]:
        parser = Sbopen()
        df= parser.event(match)[0]
        passes = df.loc[df['team_name'] == team_name].loc[df['type_name'] == 'Pass'].loc[df['sub_type_name'] != 'Throw-in'].loc[df['sub_type_name'] != 'Corner'].loc[df['sub_type_name'] != 'Free Kick'].loc[df['sub_type_name'] != 'Kick Off'].loc[df['sub_type_name'] != 'Goal Kick'].set_index('id')  
        df_pass = passes[['x', 'y', 'end_x', 'end_y','under_pressure','pass_length']]
        df_pass_long=df_pass.loc[df_pass.pass_length>10.9361]
        df_pass_not_long=df_pass.loc[df_pass.pass_length<=10.9361]
        df_pass_pressure=df_pass.loc[df_pass.under_pressure==1]
        df_pass_not_pressure=df_pass.loc[df_pass.under_pressure!=1]
        df_pass_long_pressure=df_pass_long.loc[df_pass_long.under_pressure==1]
        df_pass_long_not_pressure=df_pass_long.loc[df_pass_long.under_pressure!=1]
        df_pass_not_long_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure==1]
        df_pass_not_long_not_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure!=1]
        passes_long_pressure += len(df_pass_long_pressure)
        passes_long_not_pressure += len(df_pass_long_not_pressure)
        passes_not_long_pressure += len(df_pass_not_long_pressure)
        passes_not_long_not_pressure += len(df_pass_not_long_not_pressure)
        res = odds_ratio([[passes_long_pressure,passes_long_not_pressure],[passes_not_long_pressure,passes_not_long_not_pressure]],kind='sample')
        number = [[passes_long_pressure,passes_long_not_pressure],[passes_not_long_pressure,passes_not_long_not_pressure]]
    odds.append((team_name,number,res.statistic,res.confidence_interval(confidence_level=0.95)))
    df_odds=pd.DataFrame(odds,columns=["Equipa","Passes","UPPOR","Intervalo de confiança"])
    return df_odds

In [159]:
list_team_uppor= [uppor_team_all(team_name=team).iloc[0] for team in teams_names]


[Equipa                                                 Netherlands
 Passes                                   [[224, 1977], [114, 547]]
 UPPOR                                                     0.543656
 Intervalo de confiança    (0.4257066419171808, 0.6942841038877189)
 Name: 0, dtype: object,
 Equipa                                                     England
 Passes                                   [[393, 2851], [190, 801]]
 UPPOR                                                     0.581131
 Intervalo de confiança    (0.4805261177784716, 0.7027995888824007)
 Name: 0, dtype: object,
 Equipa                                                       Spain
 Passes                                   [[429, 2681], [191, 753]]
 UPPOR                                                     0.630844
 Intervalo de confiança    (0.5223699742944355, 0.7618438542529422)
 Name: 0, dtype: object,
 Equipa                                                      France
 Passes                                  

In [163]:
df_list_team_uppor=pd.DataFrame(list_team_uppor).sort_values(by=["UPPOR"])
df_list_team_uppor

,Equipa,Passes,UPPOR,Intervalo de confiança
0,Romania,"[[89, 742], [72, 244]]",0.406484,"(0.28854580049613393, 0.5726273730814568)"
0,Ukraine,"[[93, 1002], [49, 234]]",0.443236,"(0.3049045672437113, 0.6443266268712728)"
0,Austria,"[[240, 1216], [119, 322]]",0.534056,"(0.41519721398041287, 0.6869398698491247)"
0,Scotland,"[[199, 608], [104, 170]]",0.535014,"(0.3996055287045675, 0.716306133991542)"
0,Netherlands,"[[224, 1977], [114, 547]]",0.543656,"(0.4257066419171808, 0.6942841038877189)"
0,Slovakia,"[[138, 1232], [76, 370]]",0.545326,"(0.4027286975015097, 0.7384148859678984)"
0,Slovenia,"[[140, 728], [84, 248]]",0.567766,"(0.4178894645339946, 0.7713947521975256)"
0,Albania,"[[97, 687], [53, 217]]",0.578095,"(0.4001345802118035, 0.8352022152238912)"
0,England,"[[393, 2851], [190, 801]]",0.581131,"(0.4805261177784716, 0.7027995888824007)"
0,Hungary,"[[151, 678], [70, 185]]",0.588601,"(0.42454768854220687, 0.8160474329008608)"


In [165]:
import pickle 

In [171]:
with open('list_euro2024_matches.pkl', 'wb') as f:  # open a file
    pickle.dump(list_euro2004_matches, f) # serialize the list

In [173]:
with open('teams_names.pkl', 'wb') as f:  # open a file
    pickle.dump(teams_names, f) # serialize the list

In [175]:
with open('df_teams_matches.pkl', 'wb') as f:  # open a file
    pickle.dump(df_teams_matches, f) # serialize the list